# Рекомендательная система — модели

Данные и факторы готовы (`01_eda.ipynb`, `02_features.ipynb`). Здесь — цикл экспериментов с моделями и выбор лучшей по согласованной метрике.

**Протокол без утечки из test:**
- TRAIN — обучение (только train-окно);
- VALIDATION — выбор модели, loss, factors, regularization, epochs;
- TEST — один финальный замер выбранных конфигураций.

Тяжёлые модели считает `scripts/run_model_experiments.py` -> `reports/week3_metrics.json`; notebook читает результаты и пересчитывает baseline вживую.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
import pandas as pd
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)

from retail_recommender.config import load_config
cfg = load_config()
M = json.loads((cfg.paths.reports_dir / 'week3_metrics.json').read_text('utf-8'))
P = M['protocol']
P

{'mode': 'strong',
 'primary': 'recall@10',
 'secondary': 'ndcg@10',
 'candidate_pool': 'full train catalog (не сужается)',
 'filter_seen': True,
 'event_weights': {'view': 1, 'addtocart': 3, 'transaction': 5},
 'confidence': '1 + Σ(веса событий пары) — одинаково для ALS и BPR',
 'split': {'train_end': '2015-08-21',
  'val': ['2015-08-22', '2015-09-04'],
  'test': ['2015-09-05', '2015-09-18']},
 'n_target_users': {'val': 3850, 'test': 3432},
 'n_warm_users': {'val': 472, 'test': 495},
 'catalog_size': 213966,
 'lightfm_note': 'LightFM рассматривался как гибридный кандидат, но в текущей Windows-среде скомпилированное C-расширение нестабильно при обучении, поэтому в итоговое сравнение вошли ALS и BPR.'}

## 1. Постановка

Top-N персональных рекомендаций товаров визитору. Implicit feedback (view / addtocart / transaction), целевое действие — сильное взаимодействие (**addtocart ∪ transaction**). Основная метрика — **Recall@10 / NDCG@10**, диагностические — Coverage@10, HitRate@10, MAP@10. Уже виденные пользователем train-товары из выдачи исключаются (политика единая для всех моделей).

## 2. Train / validation / test

Скользящий вперёд временной split (зафиксирован на этапе исследования данных). Признаки товара для гибридной модели берутся as-of начала val-окна — без информации из будущего относительно момента оценки.

In [2]:
from retail_recommender.preprocessing.interactions import load_interactions
from retail_recommender.split.temporal import describe_windows
itx = load_interactions(cfg)
win = describe_windows(itx, cfg)[['window','start','end','events','users','items',
                                  'strong_events','strong_users']]
win

,window,start,end,events,users,items,strong_events,strong_users
0,train,2015-05-03,2015-08-21,2200491,1160546,213966,65850,31834
1,val,2015-08-22,2015-09-04,237616,141453,71826,7625,3850
2,test,2015-09-05,2015-09-18,224474,133797,68778,6808,3432


In [3]:
print('target-пользователи (strong):', P['n_target_users'])
print('из них warm (есть история до окна):', P['n_warm_users'])
print('каталог train:', P['catalog_size'], 'товаров')
cold_val = P['n_target_users']['val'] - P['n_warm_users']['val']
print(f"доля холодных на val: {cold_val / P['n_target_users']['val']:.1%} "
      f"(для них любая модель = fallback на популярность)")

target-пользователи (strong): {'val': 3850, 'test': 3432}
из них warm (есть история до окна): {'val': 472, 'test': 495}
каталог train: 213966 товаров
доля холодных на val: 87.7% (для них любая модель = fallback на популярность)


## 3. Baseline — контрольная точка

Три baseline, посчитанные ранее, пересчитаны в этом же протоколе (веса событий заморожены 1-3-5). Пересчёт вживую — проверка, что протокол honest.

In [4]:
from retail_recommender.evaluation.protocol import build_eval_task, evaluate_model
from retail_recommender.models import GlobalPopularity, CategoryPopularity, ItemItemCooccurrence

val_task = build_eval_task(itx, cfg, 'val', 'strong')
rows = []
for cls in (GlobalPopularity, CategoryPopularity, ItemItemCooccurrence):
    mdl = cls().fit(val_task, cfg)
    m = evaluate_model(mdl, val_task, cfg)
    rows.append({'model': mdl.name,
                 'R@10 ALL': m['ALL']['recall@10'], 'NDCG@10 ALL': m['ALL']['ndcg@10'],
                 'R@10 WARM': m['WARM']['recall@10'], 'NDCG@10 WARM': m['WARM']['ndcg@10'],
                 'cov@10': m['ALL']['coverage@10']})
baseline_live = pd.DataFrame(rows).set_index('model').round(4)
baseline_live

,R@10 ALL,NDCG@10 ALL,R@10 WARM,NDCG@10 WARM,cov@10
model,,,,,
global_popularity,0.0081,0.0072,0.0080,0.0062,0.0001
category_popularity,0.0090,0.0075,0.0148,0.0085,0.0103
item_item_cooccurrence,0.0090,0.0077,0.0149,0.0103,0.0175


In [5]:
# те же числа из сохранённого прогона экспериментов — должны совпасть
pd.DataFrame({k: v['val']['ALL'] for k, v in M['baselines'].items()}).T[
    ['recall@10','ndcg@10','coverage@10']].round(4)

,recall@10,ndcg@10,coverage@10
global_popularity,0.0081,0.0072,0.0001
category_popularity,0.0090,0.0075,0.0103
item_item_cooccurrence,0.0090,0.0077,0.0176


## 4. Какие модели сравниваем и почему

| модель | зачем |
|---|---|
| **global / category / item-item** | нижняя планка, контрольная точка |
| **ALS** (implicit) | матричная факторизация implicit feedback — должна аккуратнее ранжировать warm-пользователей, чем co-occurrence |
| **BPR** (implicit) | попарное ранжирование — доп. эксперимент |

Все модели: обучение только на train, оценка одним протоколом, **на полном train-каталоге** (candidate pool не сужаем — implicit скорит весь каталог за миллисекунды на пользователя).

> LightFM рассматривался как гибридный кандидат, но в текущей Windows-среде скомпилированное C-расширение нестабильно при обучении, поэтому в итоговое сравнение вошли ALS и BPR.

TOP-20 property-кодов товара всё равно посчитаны (`data/interim/top_property_codes.json`) — это отдельная часть анализа; они предназначались для side-features гибрида.

## 5. Эксперименты (validation)

Сетка по factors / regularization / iterations. Метрика отбора — Recall@10 (ALL / strong / val), tie-break NDCG@10.

In [6]:
def exp_table(family):
    d = M['val_experiments'][family]
    r = []
    for name, e in d.items():
        r.append({'config': name,
                  'R@10 ALL': e['ALL']['recall@10'], 'NDCG@10 ALL': e['ALL']['ndcg@10'],
                  'R@10 WARM': e['WARM']['recall@10'], 'NDCG@10 WARM': e['WARM']['ndcg@10'],
                  'cov@10': e['ALL']['coverage@10'], 'fit_sec': e.get('fit_sec')})
    return pd.DataFrame(r).set_index('config').round(4)
exp_table('als')

,R@10 ALL,NDCG@10 ALL,R@10 WARM,NDCG@10 WARM,cov@10,fit_sec
config,,,,,,
"factors=32,reg=0.01,iters=20",0.0094,0.0078,0.0184,0.0115,0.0019,126.90
"factors=32,reg=0.1,iters=20",0.0094,0.0078,0.0184,0.0112,0.0019,130.16
"factors=32,reg=1.0,iters=20",0.0094,0.0078,0.0184,0.0112,0.0019,132.00
"factors=64,reg=0.01,iters=20",0.0094,0.0078,0.0183,0.0113,0.0035,287.85
"factors=64,reg=0.1,iters=20",0.0094,0.0078,0.0183,0.0113,0.0035,161.44
"factors=64,reg=1.0,iters=20",0.0094,0.0078,0.0182,0.0113,0.0035,127.45
"factors=64,reg=0.1,iters=40",0.0095,0.0078,0.0192,0.0114,0.0035,449.81


In [7]:
exp_table('bpr')

,R@10 ALL,NDCG@10 ALL,R@10 WARM,NDCG@10 WARM,cov@10,fit_sec
config,,,,,,
"factors=32,lr=0.01,reg=0.01,iters=150",0.0091,0.0073,0.0156,0.0068,0.0117,164.95
"factors=32,lr=0.05,reg=0.01,iters=150",0.0079,0.0071,0.0064,0.0053,0.0199,150.34
"factors=64,lr=0.01,reg=0.01,iters=150",0.0096,0.0076,0.0200,0.0098,0.0106,141.91
"factors=64,lr=0.05,reg=0.01,iters=150",0.0079,0.0068,0.0066,0.0032,0.0192,272.56


## 6. Сводная таблица качества

Лучшая конфигурация каждого семейства (по validation) против лучшего baseline, на val и test.

In [8]:
sel = M['selected']
def summ(entry):
    return {'R@10 ALL': entry['ALL']['recall@10'], 'NDCG@10 ALL': entry['ALL']['ndcg@10'],
            'R@10 WARM': entry['WARM']['recall@10'], 'R@10 COLD': entry['COLD']['recall@10'],
            'MAP@10': entry['ALL']['map@10'], 'cov@10': entry['ALL']['coverage@10']}

val_rows, test_rows = {}, {}
bb = sel['best_baseline']
val_rows[f'baseline:{bb}'] = summ(M['baselines'][bb]['val'])
test_rows[f'baseline:{bb}'] = summ(M['baselines'][bb]['test'])
for fam in ('als','bpr'):
    ckey = sel[fam]
    val_rows[f'{fam}:{ckey}'] = summ(M['val_experiments'][fam][ckey])
for label, s in M['test'].items():
    test_rows[label] = summ(s)
print('VALIDATION'); display(pd.DataFrame(val_rows).T.round(4))
print('TEST'); display(pd.DataFrame(test_rows).T.round(4))

VALIDATION


,R@10 ALL,NDCG@10 ALL,R@10 WARM,R@10 COLD,MAP@10,cov@10
baseline:item_item_cooccurrence,0.0090,0.0077,0.0149,0.0081,0.0066,0.0176
"als:factors=64,reg=0.1,iters=40",0.0095,0.0078,0.0192,0.0081,0.0066,0.0035
"bpr:factors=64,lr=0.01,reg=0.01,iters=150",0.0096,0.0076,0.0200,0.0081,0.0065,0.0106


TEST


,R@10 ALL,NDCG@10 ALL,R@10 WARM,R@10 COLD,MAP@10,cov@10
baseline:item_item_cooccurrence,0.0116,0.0086,0.0221,0.0098,0.0066,0.0187
item_item_cooccurrence,0.0116,0.0086,0.0221,0.0098,0.0066,0.0187
"als[factors=64,reg=0.1,iters=40]",0.0101,0.0075,0.0122,0.0098,0.0058,0.0032
"bpr[factors=64,lr=0.01,reg=0.01,iters=150]",0.0101,0.0077,0.0118,0.0098,0.0060,0.0082


## 7. Лучшая конфигурация


In [9]:
for k in ('best_baseline','als','bpr','overall_by_val'):
    print(f'{k:16}: {sel[k]}')
print('\nкритерий отбора:', sel['criterion'])

best_baseline   : item_item_cooccurrence
als             : factors=64,reg=0.1,iters=40
bpr             : factors=64,lr=0.01,reg=0.01,iters=150
overall_by_val  : bpr:factors=64,lr=0.01,reg=0.01,iters=150

критерий отбора: max recall@10 на ALL/strong/val, tie-break ndcg@10


## 8. Финальная оценка на test

Один замер выбранных по validation конфигураций. Test для подбора гиперпараметров не использовался.

In [10]:
test_df = pd.DataFrame({k: {'R@10 ALL': v['ALL']['recall@10'],
                            'NDCG@10 ALL': v['ALL']['ndcg@10'],
                            'R@10 WARM': v['WARM']['recall@10'],
                            'NDCG@10 WARM': v['WARM']['ndcg@10'],
                            'R@10 COLD': v['COLD']['recall@10'],
                            'cov@10': v['ALL']['coverage@10']}
                        for k, v in M['test'].items()}).T.round(4)
test_df

,R@10 ALL,NDCG@10 ALL,R@10 WARM,NDCG@10 WARM,R@10 COLD,cov@10
item_item_cooccurrence,0.0116,0.0086,0.0221,0.0149,0.0098,0.0187
"als[factors=64,reg=0.1,iters=40]",0.0101,0.0075,0.0122,0.0076,0.0098,0.0032
"bpr[factors=64,lr=0.01,reg=0.01,iters=150]",0.0101,0.0077,0.0118,0.0087,0.0098,0.0082


In [11]:
base_test = M['baselines'][bb]['test']['ALL']['recall@10']
print(f'лучший baseline (test, R@10 ALL) = {base_test:.4f}\n')
for label, s in M['test'].items():
    r = s['ALL']['recall@10']; d = r - base_test
    rel = d / base_test * 100 if base_test else float('nan')
    print(f'{label:44} R@10={r:.4f}  Δ={d:+.4f} ({rel:+.1f}%)')

лучший baseline (test, R@10 ALL) = 0.0116

item_item_cooccurrence                       R@10=0.0116  Δ=+0.0000 (+0.0%)
als[factors=64,reg=0.1,iters=40]             R@10=0.0101  Δ=-0.0014 (-12.4%)
bpr[factors=64,lr=0.01,reg=0.01,iters=150]   R@10=0.0101  Δ=-0.0015 (-13.0%)


## 8b. TOP-20 property-кодов товара

Метод — частота строк в `item_properties` среди срезов `snapshot_ts <= cutoff` (не хардкод, `scripts/build_top_properties.py`). Два списка:
- **A** — cutoff `2015-07-01` (feature cutoff проекта) — основной для генератора факторов;
- **B** — cutoff `2015-08-22` (начало validation) — train-window диагностика для этого этапа.

ALS/BPR TOP-20 не использовали, поэтому списки только фиксируются.

In [12]:
tp = M['top_properties']
A, B = tp['primary'], tp['val_window_diagnostic']
print(f"A @ {A['cutoff']}: {A['top_codes']}")
print(f"B @ {B['cutoff']}: {B['top_codes']}")
print()
print('списки идентичны:', tp['lists_identical'],
      '| только в A:', tp['only_in_primary'], '| только в B:', tp['only_in_val_window'])
assert len(A['top_codes']) == 20 and len(B['top_codes']) == 20
pd.DataFrame({'A (2015-07-01)': A['top_codes'], 'B (2015-08-22)': B['top_codes']})

A @ 2015-07-01: ['888', '790', 'available', 'categoryid', '6', '283', '776', '678', '364', '159', '112', '764', '917', '202', '839', '227', '698', '689', '28', '928']
B @ 2015-08-22: ['888', '790', 'available', 'categoryid', '6', '283', '776', '678', '364', '202', '112', '764', '917', '159', '839', '227', '698', '689', '451', '663']

списки идентичны: False | только в A: ['28', '928'] | только в B: ['451', '663']


,A (2015-07-01),B (2015-08-22)
0,888,888
1,790,790
2,available,available
3,categoryid,categoryid
4,6,6
5,283,283
6,776,776
7,678,678
8,364,364
9,159,202


## 9. Выводы

**Новые модели не побили baseline на test.** По validation ALS и BPR слегка опережали co-occurrence на warm-сегменте (R@10 ~0.019–0.020 против ~0.015), и по этому критерию были выбраны. На test картина развернулась: co-occurrence — R@10 ALL **0.0116**, WARM **0.0221**; ALS — 0.0101 / 0.0122; BPR — 0.0101 / 0.0118. То есть на test ALS/BPR примерно на 12–13% хуже baseline по ALL и на ~45% хуже по WARM.

**Почему так:**
- warm-сегмент крошечный (val 472, test 495 пользователей), у каждого обычно 1–2 релевантных товара из каталога 214k — метрика на нём шумная, и зазор val→test у ALS/BPR (0.019→0.012) укладывается в этот шум;
- 86% target-пользователей холодные — для них все модели дают один и тот же popularity-fallback, поэтому разница по ALL мала и определяется меньшинством;
- ALS даёт почти одинаковый результат при любых factors/regularization (0.0094 на val) — implicit-сигнала в истории ~470 warm-пользователей мало, чтобы факторизация извлекла устойчивое ранжирование;
- BPR при learning_rate=0.05 разваливается (WARM R@10 ~0.006);
- co-occurrence выигрывает именно там, где есть короткая история (сегмент hist=2–4 на test: R@10 0.057 против 0.023 у ALS) — прямая «вместе смотрят / вместе покупают» связь на этих данных сильнее латентных факторов.

**Вывод:** на текущем объёме warm-взаимодействий матричная факторизация не даёт выигрыша над explainable-baseline; честный лучший результат на test — **item-item co-occurrence, Recall@10 = 0.0116 (WARM 0.0221)**. Резерв — контентные/сессионные подходы для холодных (LightFM здесь и задумывался, но не запустился в среде).

TOP-20 property-кодов зафиксированы в `data/interim/top_property_codes.json` — два списка (основной при feature cutoff 2015-07-01, диагностический при 2015-08-22).